# Qlib Recorder Analysis validation v1

目标：把 `github:microsoft/qlib` 外部知识候选补成 `qlib_recorder_analysis` 的 research evidence。该 notebook 只验证当前 `src/analysis` 的 recorder artifact ingestion、regime/window analysis 和 manifest export 契约，不复制 Qlib 文档或源码，不 vendoring 第三方代码。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import (
    build_qlib_recorder_regime_window_analysis_report,
    load_qlib_recorder_analysis,
    write_analysis_report,
)
from src.core import DatasetSnapshot

EXPERIMENT = "qlib_recorder_analysis_validation_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_ID = "github:microsoft/qlib"
SOURCE_URL = "https://github.com/microsoft/qlib"
RECORDER_DOC_URL = "https://qlib.readthedocs.io/en/latest/component/recorder.html"


class FakeQlibRecorder:
    def __init__(self, objects: dict[str, object]):
        self.objects = objects

    def load_object(self, name: str):
        if name not in self.objects:
            raise FileNotFoundError(name)
        return self.objects[name]


def qlib_report_from_returns(returns: list[float], benchmark: list[float]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "return": returns,
            "bench": benchmark,
            "cost": [0.0, 0.001, 0.002, 0.001, 0.002, 0.001, 0.001, 0.001],
        },
        index=pd.date_range("2026-01-01", periods=8, freq="D"),
    )


ridge = load_qlib_recorder_analysis(
    FakeQlibRecorder(
        {
            "portfolio_analysis/report_normal_1day.pkl": qlib_report_from_returns(
                [0.0, 0.02, 0.02, -0.01, -0.015, -0.01, 0.005, 0.01],
                [0.0, 0.01, 0.01, -0.01, -0.02, -0.01, 0.0, 0.0],
            ),
            "portfolio_analysis/risk_analysis_1day.pkl": {
                "annualized_return": 0.18,
                "information_ratio": 0.9,
                "max_drawdown": -0.04,
            },
        }
    ),
    model_name="ridge_mse",
)
gcn = load_qlib_recorder_analysis(
    FakeQlibRecorder(
        {
            "portfolio_analysis/report_normal_1day.pkl": qlib_report_from_returns(
                [0.0, 0.04, 0.04, -0.01, -0.06, -0.04, 0.01, 0.01],
                [0.0, 0.01, 0.01, -0.01, -0.02, -0.01, 0.0, 0.0],
            ),
            "portfolio_analysis/risk_analysis_1day.pkl": {
                "annualized_return": 0.22,
                "information_ratio": 0.7,
                "max_drawdown": -0.11,
            },
        }
    ),
    model_name="gcn_mse",
)
regime_frame = pd.DataFrame(
    {
        "date": pd.date_range("2026-01-01", periods=8, freq="D"),
        "regime": ["bull", "bull", "bull", "neutral", "stress", "stress", "stress", "downtrend"],
        "vix": [17, 18, 19, 24, 36, 34, 31, 23],
        "trend": [0.02, 0.03, 0.04, 0.0, -0.06, -0.05, -0.03, -0.02],
        "drawdown": [0.0, 0.0, 0.0, -0.01, -0.12, -0.15, -0.10, -0.04],
    }
)
windows = {
    "early": ("2026-01-01", "2026-01-04"),
    "stress_window": ("2026-01-05", "2026-01-07"),
}

report = build_qlib_recorder_regime_window_analysis_report(
    artifact_id="qlib_recorder_analysis_validation_v1",
    recorder_analyses={"ridge_mse": ridge, "gcn_mse": gcn},
    regime_frame=regime_frame,
    windows=windows,
    baseline_model="ridge_mse",
    summary={"note": "中文 review 入口：从 qlib recorder artifact 直接生成 regime/window 模型差异。"},
)

assert report.summary["status"] == "qlib_recorder_regime_window_analysis_completed"
assert report.summary["source_model_count"] == 2
assert "qlib_portfolio_report" in report.tables
assert "qlib_risk_analysis" in report.tables
assert "model_review_briefing" in report.tables
assert len(report.tables["qlib_portfolio_report"]) == 16
assert len(report.tables["qlib_risk_analysis"]) == 6
assert "stress" in "\n".join(report.tables["model_review_briefing"]["segment"].astype(str).tolist())

dataset = DatasetSnapshot(
    name="qlib_recorder_validation",
    universe=("SOXL", "SOXX"),
    start="2026-01-01",
    end="2026-01-08",
    calendar_path="data/qlib/soxl/calendars/day.txt",
    data_paths=("research/qlib_recorder_analysis_validation_v1",),
)
export_result = write_analysis_report(report, CARD, dataset=dataset)

source_provenance = {
    "schema_version": "knowledge.source_provenance.v1",
    "source_id": SOURCE_ID,
    "source_type": "github",
    "title": "Microsoft Qlib",
    "url": SOURCE_URL,
    "summary": "Qlib recorder and analyzer workflow reference for src.analysis recorder artifact ingestion.",
    "retrieved_at": "2026-06-19",
    "access_status": "metadata_accessible",
    "source_used_as": "hypothesis_only",
    "external_source_checks": {
        "repository_url": SOURCE_URL,
        "recorder_doc_url": RECORDER_DOC_URL,
        "repository_license": "MIT",
        "copied_external_code": False,
    },
    "license_note": "只保存 Qlib GitHub URL、recorder doc URL、自有摘要和验证假设；未复制文档正文；未 vendoring 第三方代码。",
}
(CARD / "source_provenance.json").write_text(
    json.dumps(source_provenance, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
(CARD / "license_review.md").write_text(
    "# License Review\n\n"
    "## 中文结论\n\n"
    f"- GitHub 来源：{SOURCE_URL}，仓库 license 复核为 MIT。\n"
    f"- Recorder 文档来源：{RECORDER_DOC_URL}，只保存 URL 和自有摘要。\n"
    "- 未复制 Qlib 文档正文，未保存未授权正文，未 vendoring Qlib 或第三方实现代码。\n"
    "- 本 notebook 只验证本仓库已有 `src.analysis.qlib_recorder` 的 artifact 标准化、regime/window 分析和 manifest export 契约。\n"
    "- 后续如要直接依赖 Qlib runtime 或扩展 recorder 对象，需要单独复核依赖版本、license 和真实 recorder artifact 兼容性。\n",
    encoding="utf-8",
)

recorder_analysis_summary = {
    "schema_version": "analysis.qlib_recorder.validation_summary.v1",
    "models": ["ridge_mse", "gcn_mse"],
    "loaded_artifacts": {
        "ridge_mse": ridge.summary["loaded_artifacts"],
        "gcn_mse": gcn.summary["loaded_artifacts"],
    },
    "table_counts": report.table_counts(),
    "window_names": report.summary["window_names"],
    "analysis_layers": report.summary["analysis_layers"],
}
(CARD / "recorder_analysis_summary.json").write_text(
    json.dumps(recorder_analysis_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

summary = {
    "schema_version": "research.qlib_recorder_analysis_validation.v1",
    "status": "completed",
    "source_id": SOURCE_ID,
    "candidate_name": "qlib_recorder_analysis",
    "target_subsystem": "src/analysis",
    "target_interface": "src.analysis.qlib_recorder + src.analysis.build_regime_window_analysis_report",
    "analysis_report_status": report.summary["status"],
    "source_model_count": report.summary["source_model_count"],
    "analysis_layers": report.summary["analysis_layers"],
    "table_counts": report.table_counts(),
    "implementation_allowed": False,
    "promotion_target": "research_only",
    "evidence_requirements": [
        "license_review",
        "research_notebook",
        "focused_pytest",
        "manifest_or_export",
        "chinese_conclusion",
    ],
    "artifacts": {
        "analysis_summary": "analysis_summary.json",
        "analysis_manifest": "analysis_manifest.json",
        "recorder_analysis_summary": "recorder_analysis_summary.json",
        "source_provenance": "source_provenance.json",
        "license_review": "license_review.md",
        **export_result["paths"],
    },
}
(CARD / "qlib_recorder_analysis_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.qlib_recorder_analysis_validation.v1',
 'status': 'completed',
 'source_id': 'github:microsoft/qlib',
 'candidate_name': 'qlib_recorder_analysis',
 'target_subsystem': 'src/analysis',
 'target_interface': 'src.analysis.qlib_recorder + src.analysis.build_regime_window_analysis_report',
 'analysis_report_status': 'qlib_recorder_regime_window_analysis_completed',
 'source_model_count': 2,
 'analysis_layers': ['model_comparison',
  'regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_return_frame',
  'qlib_risk_metrics',
  'segment_metrics',
  'segment_effect_matrix',
  'model_effect_deltas',
  'model_regime_window_diagnostics',
  'model_difference_explanations',
  'model_review_briefing',
  'failure_modes',
  'qlib_portfolio_report',
  'qlib_risk_analysis'],
 'table_counts': {'failure_modes': 4,
  'model_comparison': 2,
  'model_difference_explanations': 10,
  'model_effect_deltas': 20,
  'model_regime_window_diagnostics': 2,
  'model_revie

In [2]:
assert summary["schema_version"] == "research.qlib_recorder_analysis_validation.v1"
assert summary["candidate_name"] == "qlib_recorder_analysis"
assert summary["target_subsystem"] == "src/analysis"
assert summary["analysis_report_status"] == "qlib_recorder_regime_window_analysis_completed"
assert summary["source_model_count"] == 2
assert summary["table_counts"]["qlib_portfolio_report"] == 16
assert summary["table_counts"]["qlib_risk_analysis"] == 6
assert (CARD / "analysis_summary.json").exists()
assert (CARD / "analysis_manifest.json").exists()
assert (CARD / "source_provenance.json").exists()
assert (CARD / "license_review.md").exists()
summary


{'schema_version': 'research.qlib_recorder_analysis_validation.v1',
 'status': 'completed',
 'source_id': 'github:microsoft/qlib',
 'candidate_name': 'qlib_recorder_analysis',
 'target_subsystem': 'src/analysis',
 'target_interface': 'src.analysis.qlib_recorder + src.analysis.build_regime_window_analysis_report',
 'analysis_report_status': 'qlib_recorder_regime_window_analysis_completed',
 'source_model_count': 2,
 'analysis_layers': ['model_comparison',
  'regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_return_frame',
  'qlib_risk_metrics',
  'segment_metrics',
  'segment_effect_matrix',
  'model_effect_deltas',
  'model_regime_window_diagnostics',
  'model_difference_explanations',
  'model_review_briefing',
  'failure_modes',
  'qlib_portfolio_report',
  'qlib_risk_analysis'],
 'table_counts': {'failure_modes': 4,
  'model_comparison': 2,
  'model_difference_explanations': 10,
  'model_effect_deltas': 20,
  'model_regime_window_diagnostics': 2,
  'model_revie

## 中文结论

- 做了什么：把 `github:microsoft/qlib` 作为 `qlib_recorder_analysis` 的外部知识候选，验证当前 `src.analysis.load_qlib_recorder_analysis` 和 `build_qlib_recorder_regime_window_analysis_report` 能从 recorder-like artifact 生成稳定 analysis tables、regime/window/model review 和 manifest export。
- 核心指标：输出 `qlib_recorder_analysis_summary.json`、`recorder_analysis_summary.json`、`analysis_summary.json`、`analysis_manifest.json`、`qlib_portfolio_report.csv`、`qlib_risk_analysis.csv`、`model_review_briefing.csv`、`source_provenance.json` 和 `license_review.md`；两个 fake recorder 模型共生成 16 行 portfolio report、6 行 risk analysis，并包含 stress window 的模型差异诊断。
- regime 差异：本 notebook 使用 bull/neutral/stress/downtrend 和 early/stress_window 两个窗口验证 recorder artifact 能进入 regime/window 分层；实际生产模型表现仍以 production notebook 和 dashboard export 为准。
- 是否晋升：否。本卡只把 Qlib 外部知识从 backlog 推进到 research evidence complete，不直接进入 production，也不引入真实 Qlib runtime 依赖。
- 风险：这里使用 Qlib GitHub URL、recorder 文档 URL、MIT license 复核和自有摘要，未复制文档正文，未 vendoring Qlib 代码；fake recorder 只验证本仓库稳定接口，不能替代真实 Qlib experiment recorder 兼容性测试。
- 下一步：重新执行 `knowledge_ingest`，让 promotion audit 通过真实 research card scan 自动识别 `qlib_recorder_analysis` 的 5 项证据；后续可单独补真实 Qlib recorder artifact 回放。